# 04. Multimodal Input -- Extracting Text From an Image

This notebook (**AI-103 -> Section 01**) sends a local image, base64-encoded, alongside a text instruction to a vision-capable Azure OpenAI deployment through the Responses API's multimodal content-block format, and asks the model to extract and structure the text it finds in the image (a lightweight OCR-by-prompting use case).

**Difficulty: Intermediate**


## Prerequisites

**Python packages** (already in this repo's root `requirements.txt`):
- `openai`, `azure-identity`, `python-dotenv`
- `base64` and `pathlib` are Python standard library -- nothing extra to install.

**Azure resources**
- A **vision-capable** Azure OpenAI deployment (e.g. `gpt-4.1`, `gpt-4o`).
- Entra ID auth via `az login`.
- The image file `Agent_types.png` must exist next to this notebook -- it already does, in this same `01. Section Code/` folder.

**Environment variables** -- add to the root `.env`:
```
AZURE_OPENAI_ENDPOINT=https://<your-resource>.services.ai.azure.com/openai/v1
AZURE_OPENAI_DEPLOYMENT=gpt-4.1
```


## What You'll Learn
- The Responses API's multimodal content-block format (`input_image` / `input_text`)
- Encoding a local image as a base64 `data:` URI for inline transport
- Why notebooks need a different path-resolution strategy than the original `__file__`-based script
- A practical vision use case: reading text/diagrams out of an image


### 1. Imports and configuration


In [ ]:
import base64
import os
from pathlib import Path

from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
deployment_name = os.getenv("AZURE_OPENAI_DEPLOYMENT", "gpt-4.1")
token_provider = get_bearer_token_provider(DefaultAzureCredential(), "https://ai.azure.com/.default")


### 2. Build the client


In [ ]:
client = OpenAI(
    base_url=endpoint,
    api_key=token_provider,
)


### 3. Locate and base64-encode the image

The original script used `Path(__file__).resolve().parent` to find the image next to the `.py` file. Notebooks have no `__file__`, so we use `Path.cwd()` instead -- this assumes you launched Jupyter from (or navigated to) this notebook's own folder, which is the normal way of running notebooks in this repo (see `CLAUDE.md` -> Running Notebooks). We then read the PNG's raw bytes and base64-encode them into a plain ASCII string so they can travel inside a JSON request body.

**Exam tip:** Azure OpenAI vision-capable models accept images either as base64-encoded `data:` URIs (fully inline, nothing leaves your request) or as hosted `https://` URLs, using the same `input_image` content block either way. Base64 keeps the image entirely private to the request but inflates payload size by roughly 33% over the raw bytes -- worth remembering when working with large images or many images per request.

**Alternatives:** If your images already live in cloud storage (e.g. Azure Blob Storage), pass a hosted URL instead of re-uploading bytes on every call. For a purpose-built OCR/read pipeline instead of prompting a chat model, consider Azure AI Vision's Image Analysis / Read API, which is optimized specifically for text extraction rather than general vision reasoning.


In [ ]:
script_dir = Path.cwd()  # notebooks have no __file__; assumes you're running from this folder
image_path = script_dir / "Agent_types.png"

if not image_path.exists():
    raise FileNotFoundError(f"Image file not found: {image_path}")

print(f"Loading image from: {image_path}")
with open(image_path, "rb") as image_file:
    image_data = base64.b64encode(image_file.read()).decode("utf-8")


### 4. Send the multimodal request

The `input` here is not a plain string but a list of message dicts, each with a `role` and a `content` list mixing block types: `input_image` (the base64 data URI) and `input_text` (the instruction).

**Exam tip:** Responses API content block type names (`input_text`, `input_image`, `input_file`) differ from the Chat Completions API's block names (`text`, `image_url`). If the exam shows you a code snippet with `"type": "image_url"` inside a `messages` list, that's Chat Completions; `"type": "input_image"` inside an `input` list, as here, is the Responses API.


In [ ]:
print("Sending multimodal request to Azure OpenAI...")
response = client.responses.create(
    model=deployment_name,
    instructions="You are a helpful assistant that reads and extracts information from images.",
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_image",
                    "image_url": f"data:image/png;base64,{image_data}",
                },
                {
                    "type": "input_text",
                    "text": "Extract all the text from this image and present it in a structured, readable format.",
                },
            ],
        }
    ],
)


### 5. Read the answer


In [ ]:
print(f"answer: {response.output_text}")


## Summary

You base64-encoded a local image, sent it alongside a text instruction using the Responses API's multimodal content-block format, and printed the model's extracted/structured text.


## Try It Yourself
1. **Easy:** Point `image_path` at a different local image (e.g. one of the `.png` files in `05. Section Code/`) and re-run.
2. **Medium:** Replace the base64 data URI with a hosted image URL (same `input_image` block, just a plain `https://...` string for `image_url`) and compare.
3. **Advanced:** Loop over every `.png` in this folder, batch-OCR them, and collect the results into a `{filename: extracted_text}` dictionary.
